# Pipeline Anomaly Detector — Exploration

This notebook demonstrates end-to-end usage of the `pipeline-anomaly-detector` library:
loading fixture data, training an `EnsembleDetector`, scoring runs, and visualising results.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pipeline_anomaly_detector import PipelineRun
from pipeline_anomaly_detector.collectors.generic_collector import GenericCollector
from pipeline_anomaly_detector.features.feature_extractor import FeatureExtractor
from pipeline_anomaly_detector.features.feature_registry import FEATURE_REGISTRY
from pipeline_anomaly_detector.models.ensemble_detector import EnsembleDetector
from pipeline_anomaly_detector.models.isolation_forest_detector import IsolationForestDetector
from pipeline_anomaly_detector.models.zscore_detector import ZScoreDetector

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# Load fixture data
FIXTURES = Path('../tests/fixtures')

with (FIXTURES / 'normal_pipeline_runs.json').open() as f:
    normal_records = json.load(f)

with (FIXTURES / 'anomalous_pipeline_runs.json').open() as f:
    anomalous_records = json.load(f)

print(f'Normal runs: {len(normal_records)}')
print(f'Anomalous runs: {len(anomalous_records)}')
print(f'\nSample normal run keys: {list(normal_records[0].keys())}')

In [ ]:
# Create PipelineRun objects
normal_runs = [PipelineRun.model_validate(r) for r in normal_records]
anomalous_runs = [PipelineRun.model_validate(r) for r in anomalous_records]

all_runs = normal_runs + anomalous_runs
is_anomaly_label = [False] * len(normal_runs) + [True] * len(anomalous_runs)

print(f'Total runs: {len(all_runs)}')
print(f'Pipelines in normal data: {set(r.pipeline_name for r in normal_runs)}')
print(f'Pipelines in anomalous data: {set(r.pipeline_name for r in anomalous_runs)}')

In [ ]:
# Train EnsembleDetector on normal runs
print('Training EnsembleDetector on normal data...')

detector = EnsembleDetector(
    detectors=[
        ZScoreDetector(window=30, threshold=0.5),
        IsolationForestDetector(contamination=0.05, threshold=0.5),
    ],
    weights=None,  # equal weights
    threshold=0.6,
)
detector.fit(normal_runs)

print('Training complete!')
print(f'Detector: {detector.detector_name}')
print(f'Sub-detectors: {[d.detector_name for d in detector._detectors]}')
print(f'Weights: {detector._weights}')

In [ ]:
# Score all runs (normal + anomalous)
print('Scoring all runs...')

normal_scores = detector.batch_score(normal_runs)
anomalous_scores = detector.batch_score(anomalous_runs)
all_scores = normal_scores + anomalous_scores

# Build results DataFrame
results_df = pd.DataFrame([
    {
        'run_id': s.run_id,
        'pipeline_name': s.pipeline_name,
        'anomaly_score': s.anomaly_score,
        'is_anomaly_predicted': s.is_anomaly,
        'is_anomaly_true': label,
        'contributing_features': s.contributing_features,
        'start_time': run.start_time,
    }
    for s, run, label in zip(all_scores, all_runs, is_anomaly_label)
])

results_df['start_time'] = pd.to_datetime(results_df['start_time'])
results_df = results_df.sort_values('start_time').reset_index(drop=True)

print(f'\nScored {len(results_df)} runs')
print(f'Predicted anomalies: {results_df.is_anomaly_predicted.sum()}')
detection_rate = (results_df[results_df.is_anomaly_true].is_anomaly_predicted.mean())
print(f'Detection rate on injected anomalies: {detection_rate:.1%}')

results_df[['run_id', 'pipeline_name', 'anomaly_score', 'is_anomaly_predicted', 'is_anomaly_true']].head(10)

In [ ]:
# Plot: anomaly scores over time
fig, ax = plt.subplots(figsize=(14, 5))

# Normal runs
mask_normal = ~results_df['is_anomaly_true']
ax.scatter(
    results_df.loc[mask_normal, 'start_time'],
    results_df.loc[mask_normal, 'anomaly_score'],
    c='steelblue',
    alpha=0.6,
    s=30,
    label='Normal',
    zorder=2,
)

# Anomalous runs
mask_anomaly = results_df['is_anomaly_true']
ax.scatter(
    results_df.loc[mask_anomaly, 'start_time'],
    results_df.loc[mask_anomaly, 'anomaly_score'],
    c='crimson',
    alpha=0.85,
    s=60,
    marker='^',
    label='Injected Anomaly',
    zorder=3,
)

# Threshold line
ax.axhline(
    y=0.6,
    color='darkorange',
    linestyle='--',
    linewidth=1.5,
    label='Threshold (0.6)',
    zorder=4,
)

ax.set_xlabel('Run Start Time', fontsize=12)
ax.set_ylabel('Anomaly Score', fontsize=12)
ax.set_title('Pipeline Anomaly Scores Over Time', fontsize=14, fontweight='bold')
ax.set_ylim(-0.05, 1.05)
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f'Plot shows {len(results_df)} runs across {results_df.start_time.min().date()} → {results_df.start_time.max().date()}')

In [ ]:
# Print contributing features for top anomalies
print('=== Top Anomalies by Score ===')
print()

top_anomalies = (
    results_df[results_df['is_anomaly_predicted']]
    .nlargest(10, 'anomaly_score')
    [['run_id', 'pipeline_name', 'anomaly_score', 'is_anomaly_true', 'contributing_features']]
)

for _, row in top_anomalies.iterrows():
    label = 'TRUE ANOMALY' if row['is_anomaly_true'] else 'False Positive'
    print(f"  [{label}] {row['run_id']}")
    print(f"    Pipeline:    {row['pipeline_name']}")
    print(f"    Score:       {row['anomaly_score']:.4f}")
    print(f"    Features:    {row['contributing_features']}")
    print()

# Feature frequency chart
from collections import Counter
import itertools

all_features = list(itertools.chain.from_iterable(
    results_df[results_df['is_anomaly_predicted']]['contributing_features'].tolist()
))
feature_counts = Counter(all_features)

if feature_counts:
    fig, ax = plt.subplots(figsize=(10, 4))
    features_sorted = sorted(feature_counts.keys(), key=lambda k: feature_counts[k], reverse=True)
    counts_sorted = [feature_counts[f] for f in features_sorted]
    bars = ax.bar(features_sorted, counts_sorted, color='steelblue', edgecolor='white')
    ax.set_xlabel('Feature', fontsize=11)
    ax.set_ylabel('Times in Contributing Features', fontsize=11)
    ax.set_title('Most Common Contributing Features in Anomalies', fontsize=13, fontweight='bold')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()